# DataMate inference on a fresh Google Colab

This notebook clones the current `AskAnswer` code, downloads the LoRA adapter from the original baseline repository, downloads the pinned Qwen base model, runs the complete inference pipeline on `public_test`, and keeps the submission/logs in Google Drive.

Before running:
- Select a Colab GPU runtime: L4, A100, or another bf16 GPU with at least 22 GB. T4/P100 cannot run this model.
- Put the competition data in `MyDrive/datamate/data/`, containing `public_test/` and optionally `train/`.
- The base model is about 19 GB and is downloaded to the temporary Colab disk.

The default run drafts fresh canonical views with representative rows and enables dynamic business macros. Set `LIMIT = 10` for a smoke run, or set `REUSE_LAYERS = True` for a faster public comparison using the saved view layers.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Public repositories. The current repo intentionally does not contain the large safetensors file.
CODE_URL = "https://github.com/Yelgsef/AskAnswer.git"
ASSET_URL = "https://github.com/hugebenevolence/zalo-ai-challenge-datamate-26.git"
CODE_DIR = Path("/content/zalo-ai-challenge-datamate-26")
ASSET_DIR = Path("/content/datamate-original-assets")

DRIVE_ROOT = Path("/content/drive/MyDrive/datamate")
DATA_DIR = DRIVE_ROOT / "data"
RUNS_DIR = DRIVE_ROOT / "runs"

# Main run configuration. Change these values before the inference cell if needed.
SPLIT = "public_test"
LIMIT = None                 # None = the complete split; use 10 for a smoke run
IDS = None                  # e.g. "splits/holdout600.txt" when running a train subset
REUSE_LAYERS = False        # False uses fresh view/debt drafts and representative rows
ENABLE_DYNAMIC_MACROS = True
VIEW_SAMPLE_ROWS = 10
MACRO_MAX_COUNT = 6
MACRO_REPAIRS = 2
MACRO_MAX_CONTEXT_CHARS = 24000
SAVE_PROMPTS = False
RUN_NAME = "colab_" + SPLIT + "_full"

print("Configuration loaded.")
print("Data: ", DATA_DIR)
print("Run output: ", RUNS_DIR / RUN_NAME)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None, env=None):
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

# Clone the current code repository. Existing /content clones are reused on notebook reruns.
if not (CODE_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", "main", CODE_URL, CODE_DIR])
else:
    print("Using existing code clone:", CODE_DIR)

# Install Git LFS only when the Colab image does not already provide it.
if shutil.which("git-lfs") is None:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "git-lfs"])

# Clone the original baseline without smudging every LFS object, then fetch only the LoRA adapter.
if not (ASSET_DIR / ".git").exists():
    clone_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    run(["git", "clone", "--depth", "1", ASSET_URL, ASSET_DIR], env=clone_env)
else:
    print("Using existing asset clone:", ASSET_DIR)

run(["git", "lfs", "install", "--local"], cwd=ASSET_DIR)
run(["git", "lfs", "pull", "--include=adapters/sql_rft_v3/**"], cwd=ASSET_DIR)

LORA_DIR = ASSET_DIR / "adapters" / "sql_rft_v3"
LORA_WEIGHTS = LORA_DIR / "adapter_model.safetensors"
if not LORA_WEIGHTS.exists() or LORA_WEIGHTS.stat().st_size < 1_000_000:
    raise RuntimeError(f"LoRA download is missing or incomplete: {LORA_WEIGHTS}")
print(f"LoRA ready: {LORA_WEIGHTS} ({LORA_WEIGHTS.stat().st_size / 2**20:.1f} MiB)")

In [ ]:
assert sys.version_info >= (3, 12), f"Python 3.12+ is required, got {sys.version}"
SPLIT_DIR = DATA_DIR / SPLIT
QUESTIONS = SPLIT_DIR / "questions.jsonl"
if not QUESTIONS.exists():
    raise FileNotFoundError(f"Missing {QUESTIONS}. Upload the competition data to MyDrive/datamate/data/.")
if not (SPLIT_DIR / "rulebook.md").exists():
    raise FileNotFoundError(f"Missing {SPLIT_DIR / 'rulebook.md'}")
if not (SPLIT_DIR / "data").is_dir():
    raise FileNotFoundError(f"Missing business data directory: {SPLIT_DIR / 'data'}")

questions = [json.loads(line) for line in QUESTIONS.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
businesses = sorted({q["business_id"] for q in questions})
print(f"{SPLIT}: {len(questions)} questions, {len(businesses)} businesses")
print(f"Free /content disk: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")
if shutil.disk_usage("/content").free < 25 * 2**30:
    raise RuntimeError("At least about 25 GiB of temporary disk is recommended before downloading Qwen3.5-9B.")

In [ ]:
# Install the repository and vLLM inference dependencies using the project's existing helper.
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from datamate.notebook import big_gpu, gpus, install, setup

visible_gpus = gpus()
for gpu in visible_gpus:
    print(f"GPU: {gpu.name}, {gpu.mem_gb} GiB, sm{gpu.capability[0]}{gpu.capability[1]}, bf16={gpu.bf16}")
if not big_gpu(visible_gpus):
    raise RuntimeError("Use a bf16 GPU with at least 22 GiB (L4/A100). T4/P100 cannot run this 9B pipeline.")

os.environ["DATAMATE_DATA"] = str(DATA_DIR)
os.environ["DATAMATE_RUNS"] = str(RUNS_DIR)
os.environ["DATAMATE_MODELS"] = "/content/models"
install("llm")
env = setup()
assert env.data == DATA_DIR, (env.data, DATA_DIR)
print("Dependencies installed.")

In [ ]:
# Download the pinned base model once. The helper uses Qwen/Qwen3.5-9B at the repository's pinned revision.
model_dir = env.base_model()
lora_dir = env.adapter(LORA_DIR)
print("Base model:", model_dir)
print("LoRA adapter:", lora_dir)

# After both assets exist, inference is forced offline.
os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONHASHSEED": "0",
    "VLLM_USE_FLASHINFER_SAMPLER": "0",
})
print("Model and adapter are ready; subsequent inference is offline.")

## Run inference

This uses the same GPU-adapted flags as `scripts/flags.sh`, then explicitly enables the new representative-row and dynamic-macro features. Each visible GPU gets one business shard.

In [ ]:
out = env.runs / RUN_NAME
if out.exists() and any(out.iterdir()):
    raise RuntimeError(f"Output directory is not empty: {out}. Change RUN_NAME before rerunning.")
out.mkdir(parents=True, exist_ok=True)

flags, engine = env.inference_flags()
flags += ["--view-sample-rows", str(VIEW_SAMPLE_ROWS)]
if ENABLE_DYNAMIC_MACROS:
    flags += ["--dynamic-macros", "--macro-max-count", str(MACRO_MAX_COUNT),
              "--macro-repairs", str(MACRO_REPAIRS),
              "--macro-max-context-chars", str(MACRO_MAX_CONTEXT_CHARS)]

layers = []
if REUSE_LAYERS:
    if SPLIT != "public_test":
        raise ValueError("REUSE_LAYERS is only configured for public_test.")
    layers = ["--views-from", str(CODE_DIR / "configs/public_v9")]

ids_path = None
if IDS:
    ids_path = Path(IDS)
    if not ids_path.is_absolute():
        ids_path = CODE_DIR / ids_path
    if not ids_path.exists():
        raise FileNotFoundError(ids_path)

n = len(env.gpus)
def make_cmd(i):
    cmd = [env.python, "-m", "datamate.predict",
           "--data", SPLIT_DIR,
           "--output", out / f"shard{i}",
           "--shard", f"{i}/{n}",
           "--model", model_dir,
           "--lora", lora_dir,
           "--fewshot", CODE_DIR / "assets/fewshot_bank.jsonl",
           *flags, "--engine", engine, *layers]
    if LIMIT is not None:
        cmd += ["--limit", LIMIT]
    if ids_path:
        cmd += ["--ids", ids_path]
    if SAVE_PROMPTS:
        cmd += ["--save-prompts"]
    return cmd

print(f"Running {n} shard(s) -> {out}")
print("Flags:", " ".join(str(x) for x in flags))
env.run_shards(make_cmd, n, out, every=60.0)

In [ ]:
from datamate.submission import check, merge_shards

merge_shards(out, QUESTIONS)
submission = out / "submission.jsonl"
errors = check(submission, QUESTIONS)
if errors:
    raise RuntimeError("Submission format errors: " + "; ".join(errors[:5]))

# Keep a plainly named copy beside the run directory. Both locations are on Google Drive.
drive_submission = DRIVE_ROOT / f"{RUN_NAME}_submission.jsonl"
shutil.copy2(submission, drive_submission)
print("Submission ready:", drive_submission)
print("Full run directory:", out)

In [ ]:
# Optional local scoring for a train split. public_test has no answers and is skipped.
answers = SPLIT_DIR / "answers.jsonl"
if not answers.exists():
    print("No answers.jsonl found; public-test inference is complete.")
else:
    selected = questions
    if IDS:
        wanted = {line.strip() for line in ids_path.read_text(encoding="utf-8").splitlines() if line.strip()}
        selected = [q for q in selected if q["id"] in wanted]
    if LIMIT is not None:
        selected = selected[:int(LIMIT)]
    eval_questions = out / "questions.jsonl"
    eval_questions.write_text("".join(json.dumps(q, ensure_ascii=False) + "\n" for q in selected), encoding="utf-8")
    env.run([env.python, "-m", "datamate.scoring",
             "--data", SPLIT_DIR, "--pred", submission,
             "--questions", eval_questions, "--output", out / "score.json"])
    print("Score saved to:", out / "score.json")

In [ ]:
# Compact artifact summary for the Drive run directory.
trace_files = sorted(out.glob("shard*/trace.jsonl"))
macro_files = sorted((out / "logs").glob("_macro_registry_*.json"))
view_files = sorted((out / "logs").glob("_views_*.json"))
print(f"Trace files: {len(trace_files)}")
print(f"View artifacts: {len(view_files)}")
print(f"Macro artifacts: {len(macro_files)}")
print("Open this file for submission:", drive_submission)
print("Open this directory for logs/traces:", out)